TITLE:            Calculating the averages of climate variables under SSP245, SAI 1p0, SAI 1p5. 
PROJECT:          "Climate analogs under SAI"
AUTHORS:          Ruoyu Chen
COLLABORATORS:    
DATA INPUT:       10 ensemble files from Precipitation and Temperature for each of SSP245, SAI 1p0, SAI 1p5 scenarios. 
                   
DATA OUTPUT:      Dataframe/netcdf file of the average over a selected period for Precip and Temp for the 3 climate scenarios. 
DATE:             initiated: 5 June 2026; last run: ____
OVERVIEW:         _____ 
REQUIRES:         60 files, 10 for each category. 
NOTES:            None

In [2]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns
import geopandas as gpd
import earthpy as et
import xarray as xr
# Spatial subsetting of netcdf files
import regionmask


# Plotting options
sns.set(font_scale=1.3)
sns.set_style("white")


import warnings
warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning
)

import earthpy as et

In [68]:
#Summary Data of Files
ssp245 = xr.open_dataset("/mnt/research/nasabio/data/climate/L1/SSP245/PRECT/precip_indices_001.nc")
#print(ssp245)
#SSP dataset start from 2015 and ends at 2074

sai15 = xr.open_dataset("/mnt/research/nasabio/data/climate/L1/ARISE_SAI_1p5/PRECT/precip_indices_001.nc")
#sai15
#SAI datasets start from 2035 and ends at 2069

In [8]:
#Finding the average for SSP245 precipitation data across an example ten year period. 
climate_scenario = "SSP245" #This will be part of our path name later on, so make sure it corresponds with the right folder. 
climate_variable = "PRECT" #This is the climate variable you want to get the average of, make sure it corresponds with the right folder. 
path_files = []
for each in range(10): 
    orgin = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/" + climate_variable + "/precip_indices_" + (str(each + 1)).zfill(3) + ".nc"
    file = xr.open_dataset(orgin)
    path_files.append(file)
#We added all the files to a list





time_sliced_files = []
start_year = "2020"
end_year = "2030"
#subset each dataset temporally by the year specified in start_year and end_year, and add each sliced dataset into a list. 
for each in range(10):
    sliced = path_files[each].sel(year = slice(start_year, end_year))
    time_sliced_files.append(sliced)

#print(time_sliced_files[9]['CDD'].values) #Compare these values with concat_files_avg's numbers helps us make sure that we got the average. 
#I double checked the first number of CDD variable in each of the 10 files (119, 113, 144 ... 152) and got an average of 127.9, 
#which matched concat_files_avg's first value. You can check those values below. 

#Time to average values of the 10 files from 2020 - 2030 along the time dimension, and getting their mean. 
concat_files_avg=xr.concat(time_sliced_files, dim='time').mean(dim='time')
#print(concat_files_avg['CDD'].values) #This prints out the average CDD values to double check.  

#Let's save our data somewhere. 
save_path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/ensemble_avg/"
file_name = climate_scenario + "_avg_" + climate_variable + "_values_" + str(start_year) + "-" + str(end_year)+ ".nc"

concat_files_avg.to_netcdf(save_path + file_name)

In [56]:
#Function pulling everything together for precipitation data: 
def create_avg_ncfile_prect(climate_scenario, climate_variable, start_year, end_year):
    path_files = []
    time_sliced_files = []
    # Get the list of all files and directories
    path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/" + climate_variable 
    list_of_files = os.listdir(path)
    for each in list_of_files: 
        if (each == "p95_threshold.nc"): 
            list_of_files.remove(each)
        else: 
            file = xr.open_dataset(path + "/" + each)
            path_files.append(file)

    for each in range(len(path_files)): 
        sliced = path_files[each].sel(year = slice(start_year, end_year))
        time_sliced_files.append(sliced)
    
    concat_files_avg=xr.concat(time_sliced_files, dim='time').mean(dim='time')
    
    save_path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/ensemble_avg/"
    file_name = climate_scenario + "_avg_" + climate_variable + "_values_" + str(start_year) + "-" + str(end_year)+ ".nc"
    concat_files_avg.to_netcdf(save_path + file_name)
    return concat_files_avg

In [66]:
#Function pulling everything together for precipitation data: 
def create_avg_ncfile_tsmx(climate_scenario, climate_variable, start_year, end_year):
    path_files = []
    time_sliced_files = []
    path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/" + climate_variable + "/extreme_high"
    list_of_files = os.listdir(path)
    for each in list_of_files: 
        file = xr.open_dataset(path + "/" + each)
        path_files.append(file)
    
    for each in range(len(path_files)):
        sliced = path_files[each].sel(year = slice(start_year, end_year))
        time_sliced_files.append(sliced)
    concat_files_avg=xr.concat(time_sliced_files, dim='time').mean(dim='time')
    
    save_path = "/mnt/research/nasabio/data/climate/L1/" + climate_scenario + "/ensemble_avg/"
    file_name = climate_scenario + "_avg_" + climate_variable + "_values_" + str(start_year) + "-" + str(end_year)+ ".nc"
    concat_files_avg.to_netcdf(save_path + file_name)
    return concat_files_avg

In [16]:
create_avg_ncfile_tsmx("SSP245", "TSMX", 2020, 2030)

<xarray.Dataset> Size: 17MB
Dimensions:         (year: 11, lat: 192, lon: 288)
Coordinates:
  * year            (year) int64 88B 2020 2021 2022 2023 ... 2027 2028 2029 2030
  * lat             (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon             (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
Data variables:
    annual_mean     (year, lat, lon) float32 2MB -46.9 -46.9 -46.9 ... nan nan
    frequency       (year, lat, lon) float64 5MB 4.9 4.9 4.9 4.9 ... nan nan nan
    duration        (year, lat, lon) float64 5MB 3.825 3.825 3.825 ... nan nan
    mean_intensity  (year, lat, lon) float32 2MB -36.66 -36.66 ... nan nan
    max_intensity   (year, lat, lon) float32 2MB -21.59 -21.59 ... nan nan
Attributes:
    scenario:               SSP245
    variable:               TSMX
    member:                 001
    event_type:             extreme_high
    season:                 annual
    clim_reference:         SSP245 2015-2034
    consec_days_threshold:  3
    extreme_direction:      above
    quantile:               0.9
    land_mask:              Natural Earth 110m
    threshold_method:       pooled individual members, linear detrend

In [65]:
create_avg_ncfile_prect("ARISE_SAI_1p5", "PRECT", 2055, 2065)
create_avg_ncfile_tsmx("ARISE_SAI_1p5", "TSMX", 2055, 2065) #missing 4


create_avg_ncfile_prect("ARISE_SAI_1p0", "PRECT", 2055, 2065) #Missing 8. 
create_avg_ncfile_tsmx("ARISE_SAI_1p0", "TSMX", 2055, 2065) #Missing 2 and 4. 


<xarray.Dataset> Size: 17MB
Dimensions:         (year: 11, lat: 192, lon: 288)
Coordinates:
  * year            (year) int64 88B 2055 2056 2057 2058 ... 2062 2063 2064 2065
  * lat             (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon             (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
Data variables:
    annual_mean     (year, lat, lon) float32 2MB -46.78 -46.78 ... nan nan
    frequency       (year, lat, lon) float64 5MB 5.125 5.125 5.125 ... nan nan
    duration        (year, lat, lon) float64 5MB 4.062 4.062 4.062 ... nan nan
    mean_intensity  (year, lat, lon) float32 2MB -37.66 -37.66 ... nan nan
    max_intensity   (year, lat, lon) float32 2MB -23.69 -23.69 ... nan nan
Attributes:
    scenario:               ARISE_SAI_1p0
    variable:               TSMX
    member:                 008
    event_type:             extreme_high
    season:                 annual
    clim_reference:         SSP245 2015-2034
    consec_days_threshold:  3
    extreme_direction:      above
    quantile:               0.9
    land_mask:              Natural Earth 110m
    threshold_method:       pooled individual members, linear detrend

In [27]:
create_avg_ncfile_tsmx("SSP245", "TSMX", 2015, 2034) #This finds the average of the present period in SSP245 for temp. 
create_avg_ncfile_prect("SSP245", "PRECT", 2015, 2034) #This finds the average of the present period in SSP245 for precip. 

create_avg_ncfile_tsmx("SSP245", "TSMX", 2035, 2069) #This finds the average of the future period in SSP245 for temp. 
create_avg_ncfile_prect("SSP245", "PRECT", 2035, 2069) #This finds the average of the future period in SSP245 for precip. 

<xarray.Dataset> Size: 31MB
Dimensions:  (year: 35, lat: 192, lon: 288)
Coordinates:
  * year     (year) int64 280B 2035 2036 2037 2038 2039 ... 2066 2067 2068 2069
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Data variables:
    CDD      (year, lat, lon) float32 8MB 86.5 86.5 86.5 86.5 ... 34.5 34.5 34.5
    CWD      (year, lat, lon) float32 8MB 2.8 2.8 2.8 2.8 ... 5.7 5.7 5.7 5.7
    PRCPTOT  (year, lat, lon) float32 8MB 84.06 84.06 84.06 ... 250.5 250.4
    R95pTOT  (year, lat, lon) float32 8MB 6.12 6.12 6.12 ... 35.92 35.92 35.92
Attributes:
    scenario:                  SSP245
    member:                    001
    variable:                  PRECT
    clim_reference:            SSP245 2015-2034
    wet_day_threshold_mm_day:  1.0
    source_unit_conversion:    PRECT [m/s] x 86400 x 1000 -> mm/day